# Optimization & Regularization - Practice Exercise

Hands-on exercises with regularization techniques:
1. Implement L2 regularization and batch normalization
2. Use callbacks (EarlyStopping, ReduceLROnPlateau)
3. Experiment with data augmentation
4. Compare regularization strategies effectiveness

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Load Fashion-MNIST
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_train = X_train[..., np.newaxis] / 255.0
X_test = X_test[..., np.newaxis] / 255.0

print(f"Data shape: {X_train.shape}")

# Callbacks for training control
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

## Exercise: Regularization Comparison

Build and compare multiple regularization strategies:
1. No regularization (baseline)
2. L2 regularization (weight decay)
3. Batch normalization
4. Dropout
5. L2 + Dropout + BatchNorm (combined)

In [ ]:
# Define models with different regularization strategies
models = {}

# 1. Baseline (no regularization)
models['Baseline'] = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# 2. L2 Regularization (weight decay)
models['L2 Reg'] = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dense(10, activation='softmax')
])

# 3. Batch Normalization
models['BatchNorm'] = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(256, activation=None),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dense(128, activation=None),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dense(64, activation=None),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# 4. Dropout
models['Dropout'] = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

# 5. Combined (L2 + BatchNorm + Dropout)
models['Combined'] = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(256, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation=None, kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compile and train
results = {}
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(X_train, y_train, epochs=20, batch_size=128, validation_split=0.2,
                     callbacks=[early_stop, reduce_lr], verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    results[name] = {'history': hist, 'test_acc': test_acc, 'epochs': len(hist.history['loss'])}
    print(f"{name}: Test Acc = {test_acc:.4f}, Trained for {len(hist.history['loss'])} epochs")

print("\nTraining complete!")

In [ ]:
# Analyze overfitting by plotting train vs val curves
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for idx, (name, result) in enumerate(results.items()):
    hist = result['history']
    
    # Calculate gap between train and validation (indicator of overfitting)
    train_acc = hist.history['accuracy'][-1]
    val_acc = hist.history['val_accuracy'][-1]
    gap = train_acc - val_acc
    
    axes[idx].plot(hist.history['accuracy'], label='Train', marker='o', markersize=2, alpha=0.7)
    axes[idx].plot(hist.history['val_accuracy'], label='Validation', marker='s', markersize=2, alpha=0.7)
    axes[idx].set_title(f'{name}\nTrain-Val Gap: {gap:.4f}')
    axes[idx].set_xlabel('Epoch')
    axes[idx].set_ylabel('Accuracy')
    axes[idx].legend(fontsize=8)
    axes[idx].grid(True, alpha=0.3)

# Remove extra subplot
fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

# Create summary table
print("\n" + "="*70)
print("REGULARIZATION EFFECTIVENESS SUMMARY")
print("="*70)
print(f"{'Model':<20} {'Test Acc':<12} {'Train-Val Gap':<15} {'Epochs':<10}")
print("-"*70)

for name in sorted(results.keys()):
    result = results[name]
    hist = result['history']
    train_acc = hist.history['accuracy'][-1]
    val_acc = hist.history['val_accuracy'][-1]
    gap = train_acc - val_acc
    print(f"{name:<20} {result['test_acc']:.4f}     {gap:.4f}           {result['epochs']:<10}")

print("\nInterpretation:")
print("- Smaller Train-Val Gap = Better Generalization (less overfitting)")
print("- Larger Test Accuracy = Better performance on unseen data")
print("- Fewer Epochs = Converges faster due to validation improvement slowing")